In [3]:
import os
import json
import numpy as np
import math
from torchvision import transforms
from PIL import Image
from ultralytics import YOLO

# Initialize YOLO model (replace with your initialization)
# Our latest model for empty space detection
# model = YOLO("/home/bitbots/Downloads/rcaw25/dataset/rcaw25_day_3/model/best_empty.pt")
model = YOLO("./../lego/trained_models/legoresults_2/weights/best.pt")

# Input and output folders
input_folder = './../lego/go26/images'
output_folder = './../lego/go26/json_labels_after_training'

# Ensure output folder exists
os.makedirs(output_folder, exist_ok=True)

# Load object names
objects = {
    'names': {
                0: "2x2_blue_block",
                1: "2x2_green_block",
                2: "2x2_red_block",
                3: "2x2_yellow_block",
                4: "4x2_blue_block",
                5: "4x2_green_block",
                6: "4x2_red_block",
                7: "4x2_yellow_block"
    }
}

def calculate_rotated_vertices(x_center, y_center, width, height, angle):
    """Calculate the rotated bounding box vertices given center, width, height, and angle."""
    # Half dimensions

    angle -= np.pi / 2 
    
    half_w = width / 2
    half_h = height / 2

    # Define corner points relative to center
    corners = np.array([
        [-half_w, -half_h],
        [ half_w, -half_h],
        [ half_w,  half_h],
        [-half_w,  half_h]
    ])

    # Rotation matrix
    cos_theta = np.cos(angle)
    sin_theta = np.sin(angle)
    rotation_matrix = np.array([
        [cos_theta, -sin_theta],
        [sin_theta,  cos_theta]
    ])

    # Rotate corners
    rotated_corners = np.dot(corners, rotation_matrix.T)

    # Translate back to image coordinates
    rotated_corners[:, 0] += x_center
    rotated_corners[:, 1] += y_center

    return rotated_corners.tolist()
# Process each image in the input folder
for image_name in os.listdir(input_folder):
    if image_name.endswith('.jpg') or image_name.endswith('.png'):  # Process only image files
        image_path = os.path.join(input_folder, image_name)
        output_path = os.path.join(output_folder, os.path.splitext(image_name)[0] + '.json')

        # Perform object detection using YOLO model
        results = model(image_path)
        
        # Prepare JSON data for this image
        shapes = []
        for i in range(len(results[0].obb.data)):
            x_center, y_center, width, height, rotation = results[0].obb.xywhr[i][:5].cpu().numpy()
            rotation += np.pi / 2  # Rotate by 90 degrees (pi/2 radians)
            label_id = int(results[0].obb.cls[i].item())
            label = objects['names'][label_id]

            # Calculate rotated bounding box vertices
            points = calculate_rotated_vertices(x_center, y_center, width, height, rotation)

            # Construct shape data
            shape = {
                "label": label,
                "score": None,  # Replace with actual confidence score if available
                "points": points,
                "group_id": None,
                "description": "",
                "difficult": False,
                "visibility": "1",
                "shape_type": "rotation",
                "flags": {},
                "attributes": {},
                "direction": math.degrees(rotation)  # Convert rotation to degrees for JSON
            }
            shapes.append(shape)

        # Construct JSON data
        json_data = {
            "version": "2.3.6",
            "flags": {},
            "shapes": shapes,
            "imagePath": image_name,
            "imageData": None,
            "imageHeight": results[0].obb.orig_shape[0],
            "imageWidth": results[0].obb.orig_shape[1],
            "text": ""
        }

        # Save JSON to file
        with open(output_path, 'w') as f:
            json.dump(json_data, f, indent=2, default=str)  # Use default=str to handle non-serializable types

        print(f"Processed {image_name} and saved JSON to {output_path}")



image 1/1 /home/deekshaa/Lego_labelling/scripts/../lego/go26/images/frame0052.jpg: 480x640 2 2x2_blue_blocks, 1 2x2_yellow_block, 2 4x2_blue_blocks, 50.5ms
Speed: 1.4ms preprocess, 50.5ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)
Processed frame0052.jpg and saved JSON to ./../lego/go26/json_labels_after_training/frame0052.json

image 1/1 /home/deekshaa/Lego_labelling/scripts/../lego/go26/images/frame0015.jpg: 480x640 4 2x2_blue_blocks, 3 2x2_green_blocks, 3 2x2_red_blocks, 3 4x2_blue_blocks, 1 4x2_green_block, 1 4x2_yellow_block, 44.7ms
Speed: 1.3ms preprocess, 44.7ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)
Processed frame0015.jpg and saved JSON to ./../lego/go26/json_labels_after_training/frame0015.json

image 1/1 /home/deekshaa/Lego_labelling/scripts/../lego/go26/images/frame0006.jpg: 480x640 3 2x2_blue_blocks, 3 2x2_red_blocks, 3 4x2_blue_blocks, 1 4x2_green_block, 1 4x2_yellow_block, 43.8ms
Speed: 1.2ms preprocess, 43.8ms inference, 

In [2]:
import os
import json
import numpy as np

# Input and output folders
json_folder = './../lego/json_labels_after_training'
output_folder = './../lego/labels_txt_after_training'

# Ensure output folder exists
os.makedirs(output_folder, exist_ok=True)

# Load object names (class IDs)
objects = {
                "2x2_blue_block":0,
                "2x2_green_block":1,
                "2x2_red_block":2,
                "2x2_yellow_block":3,
                "4x2_blue_block":4,
                "4x2_green_block":5,
                "4x2_red_block":6,
                "4x2_yellow_block":7
}

# Function to calculate center, width, height, and angle
def get_bbox_properties(points, image_width, image_height, angle_degrees):
    points = np.array(points)

    # Get the center of the bounding box
    x_center = np.mean(points[:, 0]) / image_width
    y_center = np.mean(points[:, 1]) / image_height

    # Get width and height (from max-min points)
    width = (np.max(points[:, 0]) - np.min(points[:, 0])) / image_width
    height = (np.max(points[:, 1]) - np.min(points[:, 1])) / image_height

    # Convert angle from degrees to radians
    angle = np.radians(angle_degrees)

    return x_center, y_center, width, height, angle, points

# Process each JSON file
for json_file in os.listdir(json_folder):
    if json_file.endswith('.json'):
        json_path = os.path.join(json_folder, json_file)
        txt_output_path = os.path.join(output_folder, os.path.splitext(json_file)[0] + '.txt')

        # Read JSON file
        with open(json_path, 'r') as f:
            data = json.load(f)

        shapes = data.get("shapes", [])
        image_width = data.get("imageWidth", 1)
        image_height = data.get("imageHeight", 1)

        yolo_lines = []
        for shape in shapes:
            label = shape["label"]
            
            angle = shape.get("direction")
            if angle is None:
                print(f"Warning: 'direction' missing in shape for {json_file}. Skipping shape.")
                break
            
            if label not in objects:
                print(f"Warning: Label {label} not in class list. Skipping...")
                continue

            class_id = objects[label]
            points = shape["points"]
            print(os.path.splitext(json_file)[0] + '.txt')
            angle = shape["direction"]

            # Get bounding box properties (center, width, height, angle)
            x_center, y_center, width, height, angle, points = get_bbox_properties(points, image_width, image_height, angle)

            # Get the 8-point rotation coordinates
            rotated_points = np.array(points)
            x_coords = rotated_points[:, 0] / image_width  # Normalize x
            y_coords = rotated_points[:, 1] / image_height  # Normalize y

            # Flatten and join the normalized coordinates for YOLO format
            yolo_line = f"{class_id} " + " ".join([f"{x:.6f} {y:.6f}" for x, y in zip(x_coords, y_coords)])

            # Append the result for this bounding box
            yolo_lines.append(yolo_line)

        # Write to YOLO OBB text file
        with open(txt_output_path, 'w') as txt_file:
            txt_file.write("\n".join(yolo_lines))

        print(f"Converted {json_file} to {txt_output_path}")


frame0132.txt
frame0132.txt
Converted frame0132.json to ./../lego/labels_txt_after_training/frame0132.txt
frame0213.txt
frame0213.txt
frame0213.txt
frame0213.txt
frame0213.txt
frame0213.txt
frame0213.txt
frame0213.txt
Converted frame0213.json to ./../lego/labels_txt_after_training/frame0213.txt
frame0039.txt
Converted frame0039.json to ./../lego/labels_txt_after_training/frame0039.txt
frame0239.txt
frame0239.txt
frame0239.txt
frame0239.txt
frame0239.txt
frame0239.txt
Converted frame0239.json to ./../lego/labels_txt_after_training/frame0239.txt
frame0054.txt
Converted frame0054.json to ./../lego/labels_txt_after_training/frame0054.txt
frame0065.txt
Converted frame0065.json to ./../lego/labels_txt_after_training/frame0065.txt
frame0232.txt
frame0232.txt
frame0232.txt
frame0232.txt
frame0232.txt
frame0232.txt
frame0232.txt
frame0232.txt
Converted frame0232.json to ./../lego/labels_txt_after_training/frame0232.txt
frame0110.txt
frame0110.txt
Converted frame0110.json to ./../lego/labels_txt